# Compare Step 2 Infill Models

This notebook compares the original paper-style `AudioMotionTransformer` mask infiller against our `AudioFIMCausalLM` on the same classic Step 2 windows:

- left token frame `t`
- predict middle token frames `t+1, t+2, t+3`
- right token frame `t+4`
- HuBERT continuous audio features sampled onto the motion-token frame timeline

Metrics included:

- token accuracy, exact frame accuracy, exact gap accuracy, per-quantizer accuracy
- decoded RVQVAE body-feature MAE/MSE
- motion-feature FID/FGD-style Frechet distance over decoded body features
- HuBERT/audio-motion energy correlation and a small-window beat-consistency proxy

Note: the Frechet metric here is a practical RVQVAE/body-feature FID. It is not guaranteed to be the paper's official FGD unless you plug in the same evaluator feature extractor used by that paper.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "motion_generation").exists() and (path / "checkpoints").exists():
            return path
    raise RuntimeError("Could not find project root containing motion_generation/ and checkpoints/.")

PROJECT_ROOT = find_project_root()
MOTION_GENERATION_DIR = PROJECT_ROOT / "motion_generation"
sys.path.insert(0, str(MOTION_GENERATION_DIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project:", PROJECT_ROOT)
print("device:", DEVICE)

## Configuration

Keep `HISTORY_FRAMES_FOR_CAUSAL = 0` for the most apples-to-apples comparison, because the original mask transformer only sees left/right boundary frames. Raise it only if you intentionally want to test the causal model with extra history context.

In [ ]:
DATA_DIR = PROJECT_ROOT / "SuSuInterActs" / "SuSuInterActs"
MOTION_TOKEN_DIR = DATA_DIR / "motion_token_data"
EVAL_SPLIT_FILE = DATA_DIR / "split" / "val_file_list.txt"
MOTION2TEXT_JSON = DATA_DIR / "text_data" / "train.json"

MASK_CKPT = PROJECT_ROOT / "checkpoints" / "mask_transformer"
CAUSAL_CKPT = PROJECT_ROOT / "checkpoints" / "audio_fim_causal"

RVQVAE_CKPT = PROJECT_ROOT / "checkpoints" / "rvqvae" / "model" / "epoch_30.pth"
RVQVAE_MEAN_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "mean.npy"
RVQVAE_STD_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "std.npy"

# Source HuBERT feature fps and raw motion fps.
AUDIO_FPS = 50.0
RAW_MOTION_FPS = 20.0

# RVQVAE token frames are usually half the raw motion frame rate:
# raw 20 fps motion -> token 10 fps when unit_length=2.
MOTION_TOKEN_UNIT_LENGTH = 2.0
MOTION_TOKEN_FPS = RAW_MOTION_FPS / MOTION_TOKEN_UNIT_LENGTH

STEP = 4
HISTORY_FRAMES_FOR_CAUSAL = 0

# Increase these when the notebook flow is confirmed.
MAX_SEQUENCES = 128
MAX_WINDOWS = 512

# Old model masked-token generation. 1 is fastest; larger values fill masks in stages.
MASK_GENERATE_STEPS = 1

# Turn off if you only want token metrics or if RVQVAE files are unavailable.
DECODE_RVQVAE = True

# If True, clips invalid generated token IDs into 0..511 for RVQVAE decoding.
# Token metrics still count invalid IDs as wrong.
CLIP_INVALID_TOKENS_FOR_DECODE = True

AUDIO_FEAT_DIR = DATA_DIR / f"audio_features_hubert_layer9_fps{int(AUDIO_FPS)}"

for label, path in {
    "data": DATA_DIR,
    "motion tokens": MOTION_TOKEN_DIR,
    "audio features": AUDIO_FEAT_DIR,
    "eval split": EVAL_SPLIT_FILE,
    "mask checkpoint": MASK_CKPT,
    "causal checkpoint": CAUSAL_CKPT,
}.items():
    print(f"{label:18s}: {path} | exists={path.exists()}")

In [ ]:
from safetensors import safe_open

from models.audio_motion_model import AudioMotionConfig, AudioMotionTransformer
from models.audio_fim_causal_model import AudioFIMCausalDataset, AudioFIMCausalLM
from scripts.train_audio_fim_causal import (
    discover_names,
    load_action_text_map,
    load_rvqvae_model_for_eval,
    load_sequences,
    motion_token_accuracy,
    read_split_file,
    decode_body_tokens_to_features,
)

def load_mask_transformer_local(ckpt_path: Path, device: torch.device) -> AudioMotionTransformer:
    config = AudioMotionConfig.from_pretrained(str(ckpt_path))
    model = AudioMotionTransformer(config)
    safetensors_path = ckpt_path / "model.safetensors"
    bin_path = ckpt_path / "pytorch_model.bin"
    if safetensors_path.exists():
        state_dict = {}
        with safe_open(str(safetensors_path), framework="pt", device="cpu") as f:
            for key in f.keys():
                state_dict[key] = f.get_tensor(key)
    elif bin_path.exists():
        state_dict = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin in {ckpt_path}")
    model.load_state_dict(state_dict, strict=True)
    return model.to(device).eval()

def require_path(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

## Load Eval Windows

The dataset object is reused for window construction. It samples HuBERT features at `token_frame_idx * AUDIO_FPS / MOTION_TOKEN_FPS`, so both models receive audio aligned to token frames.

In [ ]:
require_path(MOTION_TOKEN_DIR, "motion token dir")
require_path(AUDIO_FEAT_DIR, "audio feature dir")
require_path(EVAL_SPLIT_FILE, "eval split file")

action_text_map = load_action_text_map(str(MOTION2TEXT_JSON) if MOTION2TEXT_JSON.exists() else None)
split_names = read_split_file(str(EVAL_SPLIT_FILE))
eval_names = discover_names(MOTION_TOKEN_DIR, AUDIO_FEAT_DIR, split_names)

eval_sequences = load_sequences(
    eval_names,
    MOTION_TOKEN_DIR,
    AUDIO_FEAT_DIR,
    action_text_map,
    max_samples=MAX_SEQUENCES,
    audio_fps=AUDIO_FPS,
    motion_fps=RAW_MOTION_FPS,
    motion_token_fps=MOTION_TOKEN_FPS,
    motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
)

eval_dataset = AudioFIMCausalDataset(
    eval_sequences,
    step=STEP,
    audio_fps=AUDIO_FPS,
    motion_fps=MOTION_TOKEN_FPS,
    min_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_windows_per_sequence=None,
    seed=42,
)

num_windows = min(MAX_WINDOWS, len(eval_dataset))
window_indices = np.linspace(0, len(eval_dataset) - 1, num_windows, dtype=np.int64).tolist()

print("sequences:", len(eval_sequences))
print("windows:", len(eval_dataset))
print("selected windows:", len(window_indices))
print("motion token fps:", MOTION_TOKEN_FPS)
print("audio fps:", AUDIO_FPS)

## Load Models

In [ ]:
require_path(MASK_CKPT, "mask transformer checkpoint")
require_path(CAUSAL_CKPT, "causal AudioFIM checkpoint")

mask_model = load_mask_transformer_local(MASK_CKPT, DEVICE)
causal_model = AudioFIMCausalLM.from_pretrained(str(CAUSAL_CKPT)).to(DEVICE).eval()

print("mask params:", sum(p.numel() for p in mask_model.parameters()))
print("causal params:", sum(p.numel() for p in causal_model.parameters()))

## Prediction Adapters

`mask_model` predicts all masked middle positions from `[left] [MASK x 12] [right]`.

`causal_model` sees its compact FIM prompt and autoregressively generates the middle tokens after the `[MIDDLE_MOTION]` marker.

In [ ]:
def tensor_to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def audio5_from_example(example) -> np.ndarray:
    parts = [
        tensor_to_numpy(example.left_audio_feature),
        tensor_to_numpy(example.middle_audio_features),
        tensor_to_numpy(example.right_audio_feature),
    ]
    return np.concatenate([parts[0][None, :], parts[1], parts[2][None, :]], axis=0).astype(np.float32)

@torch.no_grad()
def predict_mask_middle(example) -> List[List[int]]:
    audio5 = audio5_from_example(example)
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1

    input_tokens = []
    for q in range(ntpf):
        input_tokens.append(int(example.left_anchor[q]) + offsets[q])
    input_tokens.extend([mask_token_id] * (3 * ntpf))
    for q in range(ntpf):
        input_tokens.append(int(example.right_anchor[q]) + offsets[q])

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio5, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)
    output = output[0].detach().cpu().tolist()

    frames = []
    for frame_idx in range(1, 4):
        frame = []
        for q in range(ntpf):
            token_id = int(output[frame_idx * ntpf + q])
            frame.append(token_id - offsets[q])
        frames.append(frame)
    return frames

@torch.no_grad()
def predict_causal_middle(example) -> List[List[int]]:
    pred = causal_model.generate_infill(
        history_motion=example.history_motion,
        left_anchor=example.left_anchor,
        right_anchor=example.right_anchor,
        middle_audio_features=example.middle_audio_features,
        left_audio_feature=example.left_audio_feature,
        right_audio_feature=example.right_audio_feature,
        history_audio_features=example.history_audio_features,
        temperature=0.0,
    )
    return [list(map(int, frame)) for frame in pred]

def sanitize_tokens_for_decode(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> List[List[int]]:
    arr = np.asarray(frames, dtype=np.int64)
    if CLIP_INVALID_TOKENS_FOR_DECODE:
        arr = np.clip(arr, 0, codebook_size - 1)
    return arr.tolist()

def invalid_token_fraction(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> float:
    arr = np.asarray(frames, dtype=np.int64)
    return float(((arr < 0) | (arr >= codebook_size)).mean()) if arr.size else 0.0

## Metric Helpers

In [ ]:
def token_metrics(gt: Sequence[Sequence[int]], pred: Sequence[Sequence[int]]) -> Dict[str, float]:
    gt_arr = np.asarray(gt, dtype=np.int64)
    pred_arr = np.asarray(pred, dtype=np.int64)
    if gt_arr.shape != pred_arr.shape:
        return {"token_acc": np.nan, "exact_frame_acc": np.nan, "exact_gap_acc": 0.0}
    matches = gt_arr == pred_arr
    metrics = {
        "token_acc": float(matches.mean()),
        "exact_frame_acc": float(matches.all(axis=1).mean()),
        "exact_gap_acc": float(matches.all()),
    }
    for q in range(gt_arr.shape[1]):
        metrics[f"q{q + 1}_acc"] = float(matches[:, q].mean())
    return metrics

def decoded_feature_metrics(gt_feat: np.ndarray, pred_feat: np.ndarray) -> Dict[str, float]:
    diff = pred_feat.astype(np.float64) - gt_feat.astype(np.float64)
    vel_gt = np.diff(gt_feat, axis=0)
    vel_pred = np.diff(pred_feat, axis=0)
    vel_diff = vel_pred.astype(np.float64) - vel_gt.astype(np.float64)
    out = {
        "body_feat_mae": float(np.mean(np.abs(diff))),
        "body_feat_mse": float(np.mean(diff ** 2)),
        "body_feat_rmse": float(np.sqrt(np.mean(diff ** 2))),
    }
    if len(vel_diff) > 0:
        out["body_velocity_mse"] = float(np.mean(vel_diff ** 2))
    else:
        out["body_velocity_mse"] = np.nan
    return out

def pearson_or_nan(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    b = np.asarray(b, dtype=np.float64).reshape(-1)
    if len(a) != len(b) or len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def local_peak_indices(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    if len(x) == 0:
        return np.asarray([], dtype=np.int64)
    if len(x) < 3:
        return np.asarray([int(np.argmax(x))], dtype=np.int64)
    threshold = np.quantile(x, 0.70)
    peaks = []
    for i in range(1, len(x) - 1):
        if x[i] >= threshold and x[i] >= x[i - 1] and x[i] >= x[i + 1]:
            peaks.append(i)
    if not peaks:
        peaks = [int(np.argmax(x))]
    return np.asarray(peaks, dtype=np.int64)

def beat_consistency_proxy(body_feat_5: np.ndarray, audio5: np.ndarray, sigma_frames: float = 1.0) -> Dict[str, float]:
    # This is a small-window HuBERT proxy, not the official paper BC.
    motion_energy = np.linalg.norm(np.diff(body_feat_5.astype(np.float64), axis=0), axis=1)
    audio_energy = np.linalg.norm(np.diff(audio5.astype(np.float64), axis=0), axis=1)
    corr = pearson_or_nan(motion_energy, audio_energy)
    motion_beats = local_peak_indices(motion_energy)
    audio_beats = local_peak_indices(audio_energy)
    if len(motion_beats) == 0 or len(audio_beats) == 0:
        bc = np.nan
    else:
        scores = []
        for audio_idx in audio_beats:
            dist = np.min(np.abs(motion_beats - audio_idx))
            scores.append(math.exp(-(float(dist) ** 2) / (2.0 * sigma_frames ** 2)))
        bc = float(np.mean(scores))
    return {"hubert_energy_corr": corr, "hubert_bc_proxy": bc}

def sqrtm_psd(mat: torch.Tensor) -> torch.Tensor:
    mat = 0.5 * (mat + mat.T)
    vals, vecs = torch.linalg.eigh(mat)
    vals = torch.clamp(vals, min=0).sqrt()
    return (vecs * vals.unsqueeze(0)) @ vecs.T

def frechet_distance(real_features: np.ndarray, fake_features: np.ndarray, eps: float = 1e-5) -> float:
    real = torch.as_tensor(real_features, dtype=torch.float64)
    fake = torch.as_tensor(fake_features, dtype=torch.float64)
    if real.ndim != 2 or fake.ndim != 2:
        raise ValueError("features must be 2D: samples x dims")
    if real.shape[0] < 2 or fake.shape[0] < 2:
        return float("nan")
    mu1 = real.mean(dim=0)
    mu2 = fake.mean(dim=0)
    x1 = real - mu1
    x2 = fake - mu2
    cov1 = (x1.T @ x1) / max(1, real.shape[0] - 1)
    cov2 = (x2.T @ x2) / max(1, fake.shape[0] - 1)
    eye = torch.eye(cov1.shape[0], dtype=torch.float64)
    cov1 = cov1 + eps * eye
    cov2 = cov2 + eps * eye
    sqrt_cov1 = sqrtm_psd(cov1)
    covmean = sqrtm_psd(sqrt_cov1 @ cov2 @ sqrt_cov1)
    diff = mu1 - mu2
    fid = diff.dot(diff) + torch.trace(cov1 + cov2 - 2.0 * covmean)
    return float(torch.clamp(fid, min=0).item())

## Optional RVQVAE Decoder

This decodes RVQ token frames into normalized body feature vectors. The Frechet/FID-style metric below is computed in this decoded feature space.

In [ ]:
rvqvae = None
rvq_mean = None
rvq_std = None

if DECODE_RVQVAE:
    require_path(RVQVAE_CKPT, "RVQVAE checkpoint")
    require_path(RVQVAE_MEAN_PATH, "RVQVAE mean")
    require_path(RVQVAE_STD_PATH, "RVQVAE std")
    rvqvae, rvq_config = load_rvqvae_model_for_eval(RVQVAE_CKPT, DEVICE)
    rvq_mean = torch.tensor(np.load(RVQVAE_MEAN_PATH), dtype=torch.float32, device=DEVICE)
    rvq_std = torch.tensor(np.load(RVQVAE_STD_PATH), dtype=torch.float32, device=DEVICE)
    print("loaded RVQVAE:", RVQVAE_CKPT)
else:
    print("RVQVAE decoding disabled.")

@torch.no_grad()
def decode_body_features(frames: Sequence[Sequence[int]]) -> np.ndarray:
    if rvqvae is None:
        raise RuntimeError("RVQVAE decoding is disabled.")
    clean = sanitize_tokens_for_decode(frames)
    feat = decode_body_tokens_to_features(rvqvae, clean, rvq_mean, rvq_std, DEVICE)
    return feat.detach().cpu().numpy().astype(np.float32)

## Run Comparison

In [ ]:
rows = []
fid_buckets = {
    "mask": {"gt": [], "pred": []},
    "causal": {"gt": [], "pred": []},
}

for idx in tqdm(window_indices, desc="compare windows"):
    example = eval_dataset[int(idx)]
    gt_middle = [list(map(int, frame)) for frame in example.middle_motion]
    full_gt = [list(example.left_anchor)] + gt_middle + [list(example.right_anchor)]
    audio5 = audio5_from_example(example)

    predictions = {
        "mask": predict_mask_middle(example),
        "causal": predict_causal_middle(example),
    }

    decoded_gt_middle = None
    decoded_gt_full = None
    if DECODE_RVQVAE:
        decoded_gt_middle = decode_body_features(gt_middle)
        decoded_gt_full = decode_body_features(full_gt)

    for model_name, pred_middle in predictions.items():
        row = {
            "model": model_name,
            "dataset_idx": int(idx),
            "name": example.name,
            "left_idx": int(example.left_idx),
            "right_idx": int(example.right_idx),
            "invalid_token_frac": invalid_token_fraction(pred_middle),
        }
        row.update(token_metrics(gt_middle, pred_middle))

        if DECODE_RVQVAE:
            pred_middle_clean = sanitize_tokens_for_decode(pred_middle)
            full_pred = [list(example.left_anchor)] + pred_middle_clean + [list(example.right_anchor)]
            decoded_pred_middle = decode_body_features(pred_middle_clean)
            decoded_pred_full = decode_body_features(full_pred)
            row.update(decoded_feature_metrics(decoded_gt_middle, decoded_pred_middle))
            row.update(beat_consistency_proxy(decoded_pred_full, audio5))
            fid_buckets[model_name]["gt"].append(decoded_gt_middle.reshape(-1, decoded_gt_middle.shape[-1]))
            fid_buckets[model_name]["pred"].append(decoded_pred_middle.reshape(-1, decoded_pred_middle.shape[-1]))

        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.head()

## Aggregate Results

In [ ]:
mean_cols = [col for col in results_df.columns if col not in {"model", "name"}]
summary = results_df.groupby("model")[mean_cols].mean(numeric_only=True).T
display(summary)

global_rows = []
if DECODE_RVQVAE:
    for model_name, bucket in fid_buckets.items():
        gt_features = np.concatenate(bucket["gt"], axis=0)
        pred_features = np.concatenate(bucket["pred"], axis=0)
        global_rows.append({
            "model": model_name,
            "motion_feature_fid": frechet_distance(gt_features, pred_features),
            "gt_feature_samples": gt_features.shape[0],
            "feature_dim": gt_features.shape[1],
        })
global_df = pd.DataFrame(global_rows)
display(global_df)

## Inspect Best/Worst Windows

This is useful when the average metrics look okay but free-running examples still look wrong.

In [ ]:
sort_col = "token_acc"
for model_name in sorted(results_df["model"].unique()):
    print("\n", "=" * 80)
    print(model_name, "best")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=False).head(5))
    print(model_name, "worst")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=True).head(5))

## Save Metric Tables

In [ ]:
SAVE_RESULTS = False
OUT_DIR = MOTION_GENERATION_DIR / "notebooks" / "step2_metric_outputs"

if SAVE_RESULTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUT_DIR / "window_metrics.csv", index=False)
    summary.to_csv(OUT_DIR / "summary_metrics.csv")
    if len(global_df):
        global_df.to_csv(OUT_DIR / "global_fid_metrics.csv", index=False)
    print("saved to", OUT_DIR)
else:
    print("Set SAVE_RESULTS = True to write CSV outputs.")